# Risk-Controlled Momentum: Day 1 Prototype

This notebook develops the pre-registered model sequentially. Each section should be understood and checked before proceeding.

## 1. Imports and configuration

### Concept before code

An **import** makes a library's tools available in the notebook. The short names are conventional aliases: `np` for NumPy, `pd` for pandas, `plt` for Matplotlib plotting, `sns` for seaborn and `yf` for yfinance. An alias changes only how we refer to a library; it does not change the library.

A Python **dictionary** stores labelled key-value pairs. Keeping every pre-registered choice in `LOCKED_CONFIG` separates research decisions from later calculations and makes accidental parameter changes easier to notice. Percentages are decimals in calculations: 10% is `0.10`. One basis point is one ten-thousandth, so 10 basis points is `10 / 10_000 = 0.001`.

The one-day `timing_lag_days` value records the rule that a return on day `t` can use information available no later than day `t-1`. It does not perform the lag yet; the lag will be implemented and checked in the signal and volatility sections.

The final two lines turn the dictionary into a one-column table for visual inspection. Nothing in this section downloads data or calculates a strategy result.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yfinance as yf

LOCKED_CONFIG = {
    "ticker": "SPY",
    "sample_start": "2007-01-01",
    "sample_end": "2025-12-31",
    "frequency": "daily",
    "price_field": "adjusted",
    "return_type": "simple",
    "momentum_lookback_days": 252,
    "volatility_window_days": 21,
    "annualisation_days": 252,
    "target_volatility": 0.10,
    "maximum_exposure": 1.5,
    "rebalancing": "daily",
    "transaction_cost_bps": 10,
    "transaction_cost_rate": 10 / 10_000,
    "cash_return": 0.0,
    "timing_lag_days": 1,
}

config_table = pd.Series(LOCKED_CONFIG, name="Locked Day 1 value").to_frame()
config_table

## 2. Download and validate adjusted price data

### Concept before code

A **DataFrame** is a labelled table with rows and columns. `yf.download(...)` returns a DataFrame containing daily market fields. A **Series** is one labelled column; after inspecting the download, we select only the adjusted SPY closing-price series needed by this model.

SPY pays distributions. An unadjusted close can fall when cash leaves the fund even though an investor received that cash. Because the locked specification requires an adjusted price, we explicitly set `auto_adjust=True`. In installed `yfinance 1.6.0`, this adjusts the OHLC fields and places the adjusted closing price in `Close`; a separate `Adj Close` column is therefore absent.

The provider treats `start` as inclusive and `end` as exclusive. We convert the locked end-date string to a pandas `Timestamp`, add a one-day `Timedelta`, and convert it back to a date string. Requesting an exclusive boundary of `2026-01-01` is how we include the locked final date `2025-12-31`; it does not extend the research sample.

The validation dictionary stores named True/False checks. We require a datetime index, increasing and unique dates, no missing selected prices, positive prices, and coverage contained within both ends of the locked sample. The seven-calendar-day tolerance recognises that 1 January or 31 December can be a weekend or market holiday; it is a data-coverage check, not a model parameter. If any check is False, `raise ValueError(...)` stops the notebook instead of allowing bad data to flow into the strategy.

In [ ]:
sample_start = pd.Timestamp(LOCKED_CONFIG["sample_start"])
sample_end = pd.Timestamp(LOCKED_CONFIG["sample_end"])
download_end_exclusive = (sample_end + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

raw_spy = yf.download(
    tickers=LOCKED_CONFIG["ticker"],
    start=LOCKED_CONFIG["sample_start"],
    end=download_end_exclusive,
    interval="1d",
    auto_adjust=True,
    actions=False,
    keepna=False,
    progress=False,
    multi_level_index=False,
)

if raw_spy.empty:
    raise RuntimeError("SPY download returned no observations.")

if "Close" not in raw_spy.columns:
    raise ValueError(f"Adjusted Close field not found. Returned columns: {raw_spy.columns.tolist()}")

spy_price = raw_spy["Close"].rename("adjusted_price")

validation_checks = {
    "datetime_index": isinstance(spy_price.index, pd.DatetimeIndex),
    "dates_in_increasing_order": spy_price.index.is_monotonic_increasing,
    "dates_are_unique": not spy_price.index.has_duplicates,
    "no_missing_prices": not spy_price.isna().any(),
    "all_prices_are_positive": spy_price.gt(0).all(),
    "starts_inside_locked_sample": spy_price.index.min() >= sample_start,
    "ends_inside_locked_sample": spy_price.index.max() <= sample_end,
    "covers_locked_sample_start": spy_price.index.min() <= sample_start + pd.Timedelta(days=7),
    "covers_locked_sample_end": spy_price.index.max() >= sample_end - pd.Timedelta(days=7),
}

failed_checks = [name for name, passed in validation_checks.items() if not passed]
if failed_checks:
    raise ValueError(f"SPY data validation failed: {failed_checks}")

data_summary = pd.Series(
    {
        "ticker": LOCKED_CONFIG["ticker"],
        "selected_field": spy_price.name,
        "first_observation": spy_price.index.min().date(),
        "last_observation": spy_price.index.max().date(),
        "observations": spy_price.size,
        "missing_prices": int(spy_price.isna().sum()),
        "download_end_exclusive": download_end_exclusive,
    },
    name="Validated value",
)

display(data_summary.to_frame())
display(pd.Series(validation_checks, name="Passed").to_frame())
display(spy_price.head(3).to_frame())
display(spy_price.tail(3).to_frame())

## 3. Calculate simple daily returns

### Concept before code

A **simple return** measures the fractional change from one adjusted closing price to the next: `return[t] = price[t] / price[t-1] - 1`. If price rises from 100 to 102, the simple return is `0.02`, which means 2%. If it falls from 100 to 98, the return is `-0.02`, or -2%. Returns are stored as decimals, not as numbers already multiplied by 100.

`pct_change()` applies this calculation between consecutive rows. Despite its name, pandas returns a fractional change: `0.02`, not `2`. We explicitly use `fill_method=None` so a missing price would not be silently filled before the calculation.

The first return is necessarily `NaN` (not a number) because the first in-sample price has no earlier in-sample price. We retain the same index as the price series, so the return labelled `2007-01-04` represents the movement from the close on `2007-01-03` to the close on `2007-01-04`.

Validation checks require identical price/return dates, exactly one missing value in the first position, no later missing or infinite values, and agreement between `pct_change()` and one return calculated directly from the locked formula.

In [ ]:
spy_return = spy_price.pct_change(fill_method=None).rename("asset_return")

manual_second_return = spy_price.iloc[1] / spy_price.iloc[0] - 1
usable_returns = spy_return.dropna()

return_validation_checks = {
    "same_index_as_prices": spy_return.index.equals(spy_price.index),
    "same_number_of_rows_as_prices": spy_return.size == spy_price.size,
    "first_return_is_missing": pd.isna(spy_return.iloc[0]),
    "exactly_one_missing_return": spy_return.isna().sum() == 1,
    "no_missing_returns_after_first": spy_return.iloc[1:].notna().all(),
    "all_usable_returns_are_finite": np.isfinite(usable_returns).all(),
    "second_return_matches_formula": np.isclose(spy_return.iloc[1], manual_second_return),
}

failed_return_checks = [
    name for name, passed in return_validation_checks.items() if not passed
]
if failed_return_checks:
    raise ValueError(f"SPY return validation failed: {failed_return_checks}")

return_summary = pd.Series(
    {
        "price_observations": spy_price.size,
        "usable_return_observations": usable_returns.size,
        "missing_returns": int(spy_return.isna().sum()),
        "first_usable_return_date": usable_returns.index.min().date(),
        "minimum_daily_return": usable_returns.min(),
        "maximum_daily_return": usable_returns.max(),
    },
    name="Validated value",
)

return_preview = pd.concat([spy_price, spy_return], axis=1).head(4)
manual_example = pd.Series(
    {
        "previous_adjusted_price": spy_price.iloc[0],
        "current_adjusted_price": spy_price.iloc[1],
        "manual_simple_return": manual_second_return,
        "pct_change_return": spy_return.iloc[1],
    },
    name=usable_returns.index[0].date(),
)

display(return_summary.to_frame())
display(pd.Series(return_validation_checks, name="Passed").to_frame())
display(return_preview)
display(manual_example.to_frame())

## 4. Build the buy-and-hold benchmark

### Concept before code

A **portfolio weight** describes the fraction of capital exposed to an asset. Buy and hold uses a constant SPY weight of `1.0`, meaning 100% exposure. Its daily return is therefore `1.0 * asset_return`, so it should match SPY's adjusted-price return on every usable date. This section builds a gross benchmark and does not introduce transaction costs.

A **wealth index** answers what one unit of starting capital would become. Wealth compounds multiplicatively: `wealth[t] = wealth[t-1] * (1 + return[t])`. The daily value `1 + return` is a growth factor; a 2% return produces a factor of `1.02`, while a -2% return produces `0.98`. `cumprod()` multiplies those growth factors through time.

The first benchmark return remains `NaN`, because it is still undefined. For the wealth calculation only, we set the first growth factor to `1.0` to represent initial wealth before any observed return. This does not reclassify the missing return as an observed zero return.

Because the returns came from one adjusted-price series, a correctly compounded buy-and-hold wealth index must equal `adjusted_price / first_adjusted_price`. That independent equivalence is our strongest benchmark check.

In [ ]:
buy_hold_exposure = pd.Series(1.0, index=spy_price.index, name="buy_hold_exposure")
buy_hold_return = (buy_hold_exposure * spy_return).rename("buy_hold_return")

buy_hold_growth = (1.0 + buy_hold_return).rename("buy_hold_growth")
buy_hold_growth.iloc[0] = 1.0
buy_hold_wealth = buy_hold_growth.cumprod().rename("buy_hold_wealth")
normalized_adjusted_price = (spy_price / spy_price.iloc[0]).rename(
    "normalized_adjusted_price"
)

buy_hold_validation_checks = {
    "exposure_is_always_one": buy_hold_exposure.eq(1.0).all(),
    "first_benchmark_return_is_missing": pd.isna(buy_hold_return.iloc[0]),
    "usable_returns_match_asset": np.allclose(
        buy_hold_return.iloc[1:], spy_return.iloc[1:]
    ),
    "initial_wealth_is_one": np.isclose(buy_hold_wealth.iloc[0], 1.0),
    "wealth_has_no_missing_values": buy_hold_wealth.notna().all(),
    "wealth_is_positive_and_finite": (
        buy_hold_wealth.gt(0).all() and np.isfinite(buy_hold_wealth).all()
    ),
    "wealth_matches_normalized_price": np.allclose(
        buy_hold_wealth, normalized_adjusted_price
    ),
}

failed_buy_hold_checks = [
    name for name, passed in buy_hold_validation_checks.items() if not passed
]
if failed_buy_hold_checks:
    raise ValueError(f"Buy-and-hold validation failed: {failed_buy_hold_checks}")

buy_hold_summary = pd.Series(
    {
        "constant_exposure": buy_hold_exposure.iloc[-1],
        "initial_wealth": buy_hold_wealth.iloc[0],
        "ending_wealth": buy_hold_wealth.iloc[-1],
        "cumulative_return": buy_hold_wealth.iloc[-1] - 1.0,
        "first_date": buy_hold_wealth.index.min().date(),
        "last_date": buy_hold_wealth.index.max().date(),
    },
    name="Validated value",
)

benchmark_preview = pd.concat(
    [spy_price, spy_return, buy_hold_exposure, buy_hold_return, buy_hold_wealth],
    axis=1,
).head(4)
wealth_equivalence_preview = pd.concat(
    [buy_hold_wealth, normalized_adjusted_price], axis=1
).tail(3)

display(buy_hold_summary.to_frame())
display(pd.Series(buy_hold_validation_checks, name="Passed").to_frame())
display(benchmark_preview)
display(wealth_equivalence_preview)

## 5. Build and lag the momentum signal

### Concept before code

The locked **momentum lookback** is 252 trading observations, roughly one trading year but not 252 calendar days. The trailing momentum value is `price[t] / price[t-252] - 1`. A positive value means the adjusted price is above its level 252 trading days earlier. The first 252 momentum values are necessarily missing because the full lookback does not yet exist.

The raw signal is binary: `1.0` (long SPY) when trailing momentum is positive and `0.0` (cash) otherwise. We deliberately preserve unavailable momentum values as `NaN`; converting those early rows automatically to cash would pretend that the strategy made a decision before it had enough information.

Momentum calculated at the close of day `t` uses `price[t]`, so it becomes known only when return `t` has already occurred. It cannot be multiplied by that same day's return. `shift(1)` moves the raw signal from row `t-1` onto row `t`, giving the implementable position that earns return `t`. This is the key defence against look-ahead bias.

The **unscaled momentum return** is `position[t] * asset_return[t]`. A position of `1.0` earns SPY's return; a position of `0.0` earns the locked Day 1 cash return of zero. No volatility estimate or leverage is used yet.

In [ ]:
momentum_lookback = LOCKED_CONFIG["momentum_lookback_days"]
trailing_momentum = spy_price.pct_change(
    periods=momentum_lookback, fill_method=None
).rename("trailing_momentum")

raw_momentum_signal = (
    trailing_momentum.gt(0).astype(float).where(trailing_momentum.notna())
).rename("raw_momentum_signal")
momentum_position = raw_momentum_signal.shift(1).rename("momentum_position")
unscaled_momentum_return = (momentum_position * spy_return).rename(
    "unscaled_momentum_return"
)

manual_first_momentum = (
    spy_price.iloc[momentum_lookback] / spy_price.iloc[0] - 1
)
valid_positions = momentum_position.dropna()
long_days = momentum_position.eq(1.0)
cash_days = momentum_position.eq(0.0)

momentum_validation_checks = {
    "first_252_momentum_values_are_missing": (
        trailing_momentum.isna().sum() == momentum_lookback
    ),
    "raw_signal_preserves_unavailable_history": (
        raw_momentum_signal.isna().sum() == momentum_lookback
    ),
    "position_has_one_additional_lagged_missing_value": (
        momentum_position.isna().sum() == momentum_lookback + 1
    ),
    "raw_signal_is_binary_when_available": set(
        raw_momentum_signal.dropna().unique()
    ).issubset({0.0, 1.0}),
    "position_is_binary_when_available": set(valid_positions.unique()).issubset(
        {0.0, 1.0}
    ),
    "position_equals_previous_raw_signal": np.allclose(
        momentum_position.to_numpy(),
        raw_momentum_signal.shift(1).to_numpy(),
        equal_nan=True,
    ),
    "first_momentum_matches_manual_formula": np.isclose(
        trailing_momentum.iloc[momentum_lookback], manual_first_momentum
    ),
    "strategy_return_missingness_matches_position": (
        unscaled_momentum_return.isna().sum() == momentum_lookback + 1
    ),
    "long_days_match_asset_return": np.allclose(
        unscaled_momentum_return.loc[long_days], spy_return.loc[long_days]
    ),
    "cash_days_earn_zero": unscaled_momentum_return.loc[cash_days].eq(0.0).all(),
}

failed_momentum_checks = [
    name for name, passed in momentum_validation_checks.items() if not passed
]
if failed_momentum_checks:
    raise ValueError(f"Momentum validation failed: {failed_momentum_checks}")

momentum_summary = pd.Series(
    {
        "lookback_observations": momentum_lookback,
        "first_raw_signal_date": raw_momentum_signal.first_valid_index().date(),
        "first_implementable_position_date": momentum_position.first_valid_index().date(),
        "implementable_days": valid_positions.size,
        "long_days": int(long_days.sum()),
        "cash_days": int(cash_days.sum()),
        "missing_strategy_returns": int(unscaled_momentum_return.isna().sum()),
    },
    name="Validated value",
)

first_position_location = spy_price.index.get_loc(momentum_position.first_valid_index())
signal_alignment_preview = pd.concat(
    [
        spy_price,
        trailing_momentum,
        raw_momentum_signal,
        momentum_position,
        spy_return,
        unscaled_momentum_return,
    ],
    axis=1,
).iloc[first_position_location - 2 : first_position_location + 3]

display(momentum_summary.to_frame())
display(pd.Series(momentum_validation_checks, name="Passed").to_frame())
display(signal_alignment_preview)

## 6. Estimate and lag rolling volatility

### Concept before code

**Volatility** measures the dispersion of returns, not their direction. A period containing large positive and negative movements can have high volatility, while small movements can have low volatility. The locked estimate is the standard deviation of the most recent 21 daily returns.

A **rolling window** advances one trading row at a time. At each date it uses the current return and the previous 20 returns, then drops the oldest return when it advances. `rolling(window=21, min_periods=21).std(ddof=1)` requires 21 valid observations and calculates sample standard deviation, using `n - 1` in the variance denominator because volatility is estimated from a finite sample.

The rolling result is daily volatility. To express it on an annual scale, we multiply by `sqrt(252)`: `annualised_volatility = daily_volatility * sqrt(252)`. The square root appears because variance approximately scales with time and standard deviation is the square root of variance. This is a conventional scaling assumption, not a guarantee about future volatility.

The estimate dated `t` includes return `t`, so it becomes known only at the close of `t`. It cannot determine the position that earned return `t`. We therefore apply `shift(1)` and use yesterday's volatility estimate for today's implementable calculation. Target exposure is deliberately left for the next section.

In [ ]:
volatility_window = LOCKED_CONFIG["volatility_window_days"]
annualisation_days = LOCKED_CONFIG["annualisation_days"]
annualisation_multiplier = np.sqrt(annualisation_days)

rolling_daily_volatility = (
    spy_return.rolling(window=volatility_window, min_periods=volatility_window)
    .std(ddof=1)
    .rename("rolling_daily_volatility")
)
raw_annualised_volatility = (
    rolling_daily_volatility * annualisation_multiplier
).rename("raw_annualised_volatility")
lagged_volatility_estimate = raw_annualised_volatility.shift(1).rename(
    "lagged_volatility_estimate"
)

first_raw_volatility_date = raw_annualised_volatility.first_valid_index()
first_raw_volatility_location = spy_return.index.get_loc(first_raw_volatility_date)
manual_first_window = spy_return.iloc[
    first_raw_volatility_location - volatility_window + 1 : first_raw_volatility_location + 1
]
manual_first_annualised_volatility = (
    manual_first_window.std(ddof=1) * annualisation_multiplier
)
available_raw_volatility = raw_annualised_volatility.dropna()
available_lagged_volatility = lagged_volatility_estimate.dropna()

volatility_validation_checks = {
    "first_21_raw_estimates_are_missing": (
        raw_annualised_volatility.isna().sum() == volatility_window
    ),
    "lag_adds_one_missing_estimate": (
        lagged_volatility_estimate.isna().sum() == volatility_window + 1
    ),
    "volatility_index_matches_returns": raw_annualised_volatility.index.equals(
        spy_return.index
    ),
    "raw_volatility_is_positive_and_finite": (
        available_raw_volatility.gt(0).all()
        and np.isfinite(available_raw_volatility).all()
    ),
    "annualisation_uses_sqrt_252": np.allclose(
        available_raw_volatility,
        rolling_daily_volatility.dropna() * np.sqrt(252),
    ),
    "lagged_estimate_equals_previous_raw_estimate": np.allclose(
        lagged_volatility_estimate.to_numpy(),
        raw_annualised_volatility.shift(1).to_numpy(),
        equal_nan=True,
    ),
    "first_estimate_matches_manual_window": np.isclose(
        raw_annualised_volatility.loc[first_raw_volatility_date],
        manual_first_annualised_volatility,
    ),
}

failed_volatility_checks = [
    name for name, passed in volatility_validation_checks.items() if not passed
]
if failed_volatility_checks:
    raise ValueError(f"Volatility validation failed: {failed_volatility_checks}")

volatility_summary = pd.Series(
    {
        "rolling_window": volatility_window,
        "annualisation_days": annualisation_days,
        "annualisation_multiplier": annualisation_multiplier,
        "first_raw_estimate_date": first_raw_volatility_date.date(),
        "first_implementable_estimate_date": (
            lagged_volatility_estimate.first_valid_index().date()
        ),
        "available_lagged_estimates": available_lagged_volatility.size,
        "minimum_lagged_volatility": available_lagged_volatility.min(),
        "median_lagged_volatility": available_lagged_volatility.median(),
        "maximum_lagged_volatility": available_lagged_volatility.max(),
    },
    name="Validated value",
)

volatility_alignment_preview = pd.concat(
    [spy_return, rolling_daily_volatility, raw_annualised_volatility, lagged_volatility_estimate],
    axis=1,
).iloc[first_raw_volatility_location - 1 : first_raw_volatility_location + 3]

display(volatility_summary.to_frame())
display(pd.Series(volatility_validation_checks, name="Passed").to_frame())
display(volatility_alignment_preview)

## 7. Apply volatility targeting and the leverage cap

### Concept before code

Volatility targeting changes the size of the position inversely with the lagged volatility estimate. When the momentum position is long, uncapped exposure is `target_volatility / lagged_volatility`. Estimated volatility above the 10% target produces exposure below `1.0`; estimated volatility below the target produces exposure above `1.0`. When the momentum position is cash, multiplying by zero keeps exposure at zero.

Examples: a 20% estimate gives `0.10 / 0.20 = 0.50` exposure; an 8% estimate gives `1.25`; and a 5% estimate gives a raw exposure of `2.0`. Exposure above `1.0` is leverage: `1.5` means 150% exposure to SPY, economically implying 50% borrowed capital. Day 1 does not model financing cost, which remains a limitation.

The locked leverage constraint is `final_exposure = min(1.5, raw_exposure)`. It prevents very low volatility estimates from creating unlimited exposure. This cap was fixed before inspecting results and is not adjusted to improve performance.

Gross volatility-targeted return is `final_exposure * asset_return`. Gross means before transaction costs. Both the momentum position and volatility estimate were already shifted, so return `t` uses only information available by the close of `t-1`. A 10% target guides position sizing; it does not guarantee 10% realised future volatility.

In [ ]:
target_volatility = LOCKED_CONFIG["target_volatility"]
maximum_exposure = LOCKED_CONFIG["maximum_exposure"]

volatility_scalar = (
    target_volatility / lagged_volatility_estimate
).rename("volatility_scalar")
raw_target_exposure = (momentum_position * volatility_scalar).rename(
    "raw_target_exposure"
)
target_exposure = raw_target_exposure.clip(upper=maximum_exposure).rename(
    "target_exposure"
)
gross_vol_targeted_return = (target_exposure * spy_return).rename(
    "gross_vol_targeted_return"
)

available_exposure = target_exposure.dropna()
available_raw_exposure = raw_target_exposure.dropna()
long_exposure_days = momentum_position.eq(1.0)
cash_exposure_days = momentum_position.eq(0.0)
leveraged_days = target_exposure.gt(1.0)
capped_days = raw_target_exposure.gt(maximum_exposure)
uncapped_days = raw_target_exposure.notna() & ~capped_days

exposure_validation_checks = {
    "exposure_index_matches_returns": target_exposure.index.equals(spy_return.index),
    "missingness_matches_momentum_position": (
        target_exposure.isna().sum() == momentum_position.isna().sum()
    ),
    "available_exposure_is_finite_and_nonnegative": (
        np.isfinite(available_exposure).all() and available_exposure.ge(0).all()
    ),
    "maximum_exposure_is_respected": available_exposure.le(
        maximum_exposure
    ).all(),
    "cash_days_have_zero_exposure": target_exposure.loc[
        cash_exposure_days
    ].eq(0.0).all(),
    "long_raw_exposure_matches_target_ratio": np.allclose(
        raw_target_exposure.loc[long_exposure_days],
        target_volatility / lagged_volatility_estimate.loc[long_exposure_days],
    ),
    "capped_days_equal_maximum_exposure": target_exposure.loc[capped_days].eq(
        maximum_exposure
    ).all(),
    "uncapped_days_equal_raw_exposure": np.allclose(
        target_exposure.loc[uncapped_days], raw_target_exposure.loc[uncapped_days]
    ),
    "gross_return_matches_exposure_times_asset": np.allclose(
        gross_vol_targeted_return.dropna(),
        (target_exposure * spy_return).dropna(),
    ),
    "cash_days_have_zero_gross_return": gross_vol_targeted_return.loc[
        cash_exposure_days
    ].eq(0.0).all(),
}

failed_exposure_checks = [
    name for name, passed in exposure_validation_checks.items() if not passed
]
if failed_exposure_checks:
    raise ValueError(f"Exposure validation failed: {failed_exposure_checks}")

exposure_summary = pd.Series(
    {
        "volatility_target": target_volatility,
        "maximum_exposure": maximum_exposure,
        "first_exposure_date": target_exposure.first_valid_index().date(),
        "implementable_exposure_days": available_exposure.size,
        "average_exposure": available_exposure.mean(),
        "median_long_exposure": target_exposure.loc[long_exposure_days].median(),
        "leveraged_days": int(leveraged_days.sum()),
        "cap_binding_days": int(capped_days.sum()),
        "maximum_observed_exposure": available_exposure.max(),
    },
    name="Validated value",
)

first_capped_date = raw_target_exposure.loc[capped_days].index[0]
first_capped_location = spy_price.index.get_loc(first_capped_date)
exposure_preview = pd.concat(
    [
        momentum_position,
        lagged_volatility_estimate,
        volatility_scalar,
        raw_target_exposure,
        target_exposure,
        spy_return,
        gross_vol_targeted_return,
    ],
    axis=1,
).iloc[first_capped_location - 2 : first_capped_location + 3]

display(exposure_summary.to_frame())
display(pd.Series(exposure_validation_checks, name="Passed").to_frame())
display(exposure_preview)

## 8. Calculate turnover and transaction costs

### Concept before code

**Turnover** measures the absolute change in portfolio exposure: `turnover[t] = abs(exposure[t] - exposure[t-1])`. Increasing and decreasing exposure both require trading, so both create positive turnover. Moving from `0.5` to `1.0` creates turnover of `0.5`; moving from `1.5` to cash creates turnover of `1.5`. Daily volatility resizing can create turnover even when the momentum signal stays long.

The pre-registered transaction-cost assumption is ten basis points per unit of turnover. One basis point is `0.0001`, so ten basis points is `10 / 10_000 = 0.001`. Turnover of `0.5` therefore costs `0.0005` of portfolio value, or 0.05%; turnover of `1.5` costs `0.0015`, or 0.15%.

The specification does not state the exposure immediately before the first implementable strategy date. We explicitly initialise previous exposure at `0.0`, meaning the portfolio starts in cash. The first move into SPY therefore creates turnover and incurs cost rather than receiving a free initial trade. This is an implementation convention, not a tuned parameter.

Net return is `gross_return - estimated_transaction_cost`. The cost model is linear and estimated: it does not separately represent spreads, market impact, taxes, financing or uncertain execution. Performance metrics are left for the next section.

In [ ]:
transaction_cost_bps = LOCKED_CONFIG["transaction_cost_bps"]
transaction_cost_rate = LOCKED_CONFIG["transaction_cost_rate"]

first_exposure_date = target_exposure.first_valid_index()
first_exposure_location = target_exposure.index.get_loc(first_exposure_date)
previous_target_exposure = target_exposure.shift(1).rename(
    "previous_target_exposure"
)
previous_target_exposure.loc[first_exposure_date] = 0.0
exposure_change = (target_exposure - previous_target_exposure).rename(
    "exposure_change"
)
turnover = exposure_change.abs().rename("turnover")
estimated_transaction_cost = (transaction_cost_rate * turnover).rename(
    "estimated_transaction_cost"
)
net_vol_targeted_return = (
    gross_vol_targeted_return - estimated_transaction_cost
).rename("net_vol_targeted_return")

available_turnover = turnover.dropna()
available_cost = estimated_transaction_cost.dropna()
turnover_days = turnover.gt(0.0)
unchanged_exposure_days = exposure_change.eq(0.0)

cost_validation_checks = {
    "ten_bps_equals_decimal_rate": np.isclose(
        transaction_cost_rate, transaction_cost_bps / 10_000
    ),
    "initial_previous_exposure_is_zero": np.isclose(
        previous_target_exposure.loc[first_exposure_date], 0.0
    ),
    "initial_turnover_equals_initial_exposure": np.isclose(
        turnover.loc[first_exposure_date], target_exposure.loc[first_exposure_date]
    ),
    "turnover_missingness_matches_exposure": (
        turnover.isna().sum() == target_exposure.isna().sum()
    ),
    "turnover_is_finite_and_nonnegative": (
        np.isfinite(available_turnover).all() and available_turnover.ge(0.0).all()
    ),
    "later_turnover_matches_absolute_weight_change": np.allclose(
        turnover.iloc[first_exposure_location + 1 :],
        target_exposure.diff().abs().iloc[first_exposure_location + 1 :],
    ),
    "cost_equals_rate_times_turnover": np.allclose(
        available_cost, transaction_cost_rate * available_turnover
    ),
    "unchanged_exposure_has_zero_cost": estimated_transaction_cost.loc[
        unchanged_exposure_days
].eq(0.0).all(),
    "positive_turnover_has_positive_cost": estimated_transaction_cost.loc[
        turnover_days
].gt(0.0).all(),
    "net_return_identity_holds": np.allclose(
        net_vol_targeted_return.dropna(),
        (gross_vol_targeted_return - estimated_transaction_cost).dropna(),
    ),
    "net_return_never_exceeds_gross_return": (
        net_vol_targeted_return.dropna()
        .le(gross_vol_targeted_return.dropna())
        .all()
    ),
}

failed_cost_checks = [
    name for name, passed in cost_validation_checks.items() if not passed
]
if failed_cost_checks:
    raise ValueError(f"Turnover and cost validation failed: {failed_cost_checks}")

cost_summary = pd.Series(
    {
        "transaction_cost_bps": transaction_cost_bps,
        "transaction_cost_rate": transaction_cost_rate,
        "initial_exposure": target_exposure.loc[first_exposure_date],
        "initial_turnover": turnover.loc[first_exposure_date],
        "initial_transaction_cost": estimated_transaction_cost.loc[
            first_exposure_date
        ],
        "days_with_turnover": int(turnover_days.sum()),
        "total_turnover": available_turnover.sum(),
        "average_daily_turnover": available_turnover.mean(),
        "maximum_daily_turnover": available_turnover.max(),
        "total_estimated_cost": available_cost.sum(),
        "maximum_daily_cost": available_cost.max(),
    },
    name="Validated value",
)

largest_turnover_date = turnover.idxmax()
largest_turnover_location = target_exposure.index.get_loc(largest_turnover_date)
cost_preview = pd.concat(
    [
        previous_target_exposure,
        target_exposure,
        exposure_change,
        turnover,
        estimated_transaction_cost,
        gross_vol_targeted_return,
        net_vol_targeted_return,
    ],
    axis=1,
).iloc[largest_turnover_location - 2 : largest_turnover_location + 3]

display(cost_summary.to_frame())
display(pd.Series(cost_validation_checks, name="Passed").to_frame())
display(cost_preview)

## 9. Calculate performance metrics

## 10. Visualise preliminary results

## 11. Run sanity checks and record observations